In [1]:
import cvxpy as cp
import gurobipy as gp
import heapq
import itertools
import numpy as np
import pandas as pd
import time
import tqdm
import warnings

from gurobipy import GRB
warnings.filterwarnings("ignore")

In [2]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors, dtype=float) / len(priors)
    return priors / np.sum(priors)

def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + ((len(search_space)-i)*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump], axis=1)
    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i) * 1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])
    return X_p

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true, return_breakdowns=False):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    if return_breakdowns:
        return np.dot(losses, posteriors), losses * priors
    else:
        return np.dot(losses, posteriors)

def evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns=False):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true, return_breakdowns)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c, return_breakdowns=False):
    acc_loss = 0.0
    acc_loss_list = []
    for partition in partitions:
        if return_breakdowns:
            acc_loss_p, acc_loss_p_list = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
            acc_loss_list.append(acc_loss_p_list.tolist())
        else:
            acc_loss_p = evaluate_conditional(X, partition, thresholds, priors, threshold_true, c, return_breakdowns)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    if return_breakdowns:
        return acc_loss, acc_loss_list
    else:
        return acc_loss

In [3]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return
    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    t0 = time.perf_counter()
    indices = list(range(len(thresholds)))
    best_partition, best_loss = None, np.inf
    for partition in set_partitions(indices):
        acc_loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
        if acc_loss <= best_loss:
            if best_loss - acc_loss < 1e-9:
                if len(partition) < len(best_partition):
                    best_loss = acc_loss
                    best_partition = partition
            else:
                best_loss = acc_loss
                best_partition = partition
    
    return best_partition, best_loss, time.perf_counter()-t0

In [4]:
def find_partitions_greedy_agg(X, thresholds, priors, threshold_true, c, eps=1e-9):
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_conditional(X, block, thresholds, priors, threshold_true, c)
                + evaluate_conditional(X_eps, block, thresholds, priors, threshold_true, c) * eps)
    t0 = time.perf_counter()
    P = {}
    next_id = 0
    for i in range(len(priors)):
        P[next_id] = [i]
        next_id += 1

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = block_cost(ab) * np.sum(priors[ab])
        acc_loss_a  = block_cost(a)  * np.sum(priors[a])
        acc_loss_b  = block_cost(b)  * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P or b_id not in P:
            continue
        a, b = P[a_id], P[b_id]
        ab = sorted(a + b)

        acc_loss_a  = block_cost(a)
        acc_loss_b  = block_cost(b)
        acc_loss_ab = block_cost(ab)

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]
            pq = [(g, (x, y)) for g, (x, y) in pq if x not in {a_id, b_id} and y not in {a_id, b_id}]
            heapq.heapify(pq)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P:
                if p_id == new_id:
                    continue
                p = P[p_id]
                merged = sorted(ab + p)
                acc_loss_merged = block_cost(merged) * np.sum(priors[merged])
                acc_loss_p      = block_cost(p) * np.sum(priors[p])
                acc_loss_ab_new = block_cost(ab) * np.sum(priors[ab])
                gain = -(acc_loss_p + acc_loss_ab_new - acc_loss_merged)
                heapq.heappush(pq, (gain, (new_id, p_id)))
    partition = list(P.values())
    loss = evaluate_system(X, partition, thresholds, priors, threshold_true, c)
    return partition, loss, time.perf_counter()-t0

def split_partition(partition):
    idx_large = 0
    block_large = partition[idx_large]
    block_others = [partition[i] for i in range(len(partition)) if i != idx_large]
    result = []
    for part in itertools.combinations(block_large, len(block_large)-1):
        A = list(part)
        B = [x for x in block_large if x not in A]
        result.append([A] + [B] + block_others)

        for i, block in enumerate(block_others):
            merged = sorted(B + block)
            other_remaining = [block_others[j] for j in range(len(block_others)) if j != i]
            result.append([A] + [merged] + other_remaining)
    return result

def find_partitions_greedy_div(X, thresholds, priors, threshold_true, c, eps=1e-9, show_steps=False):
    n = len(priors)
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)
    
    def block_cost(block):
        return (evaluate_conditional(X, block, thresholds, priors, threshold_true, c)
                + evaluate_conditional(X_eps, block, thresholds, priors, threshold_true, c) * eps)

    t0 = time.perf_counter()
    partition_0 = [list(range(n))]
    acc_loss_0 = block_cost(partition_0[0])
    pq = [(acc_loss_0, partition_0)]

    while pq:
        acc_loss_merged, partition_merged = heapq.heappop(pq)
        if len(partition_merged[0])==1:
            loss = evaluate_system(X, partition_merged, thresholds, priors, threshold_true, c)
            return partition_merged, loss, time.perf_counter()-t0
        partitions = split_partition(partition_merged)
        pq = []
        for partition in partitions:
            acc_loss_split = 0.
            for block in partition:
                acc_loss_split += block_cost(block) * np.sum(priors[block])
            gain = acc_loss_merged - acc_loss_split
            if gain > -1e-9:
                heapq.heappush(pq, (acc_loss_split, partition))
        if not pq:
            loss = evaluate_system(X, partition_merged, thresholds, priors, threshold_true, c)
            return partition_merged, loss, time.perf_counter()-t0

def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [5]:
def validate(priors, thresholds, tt, c):
    p = np.asarray(priors, dtype=float)
    t = np.asarray(thresholds, dtype=float)
    if p.shape != t.shape or p.ndim != 1:
        raise ValueError("priors and thresholds must be 1-D of equal length")
    if np.any(p < 0):
        raise ValueError("priors must be nonnegative")
    if not np.isclose(p.sum(), 1.0):
        raise ValueError(f"priors must sum 1, got {p.sum()}")
    if c <= 0:
        raise ValueError("c must be positive")
    
    return p, t, float(tt), float(c)

def accept_matrix(thresholds):
    t = np.asarray(thresholds, dtype=float)
    return (t[None, :] >= t[:, None]).astype(float)

def build_grid(m, lo=0.0, hi=1.0):
    X = np.linspace(lo, hi, m)
    D = np.full(m, 1.0/m)
    return X, D

def build_constants(priors, thresholds, tt, c, m, eps=1e-6):
    priors, thresholds, tt, c = validate(priors, thresholds, tt, c)
    n = len(thresholds)
    X, D = build_grid(m)

    A = np.empty((m, n+1))
    A[:, 0] = X
    A[:, 1:] = thresholds[None, :]

    cost = c * np.abs(A - X[:, None])
    Hx = (A[:, None, :] >= thresholds[None, :, None]).astype(float)
    U = priors[None, :, None] * (Hx - (1.0 + eps) * cost[:, None, :])
    f = (X >= tt).astype(float)
    L = np.where(f[:, None, None] == 0.0, Hx, 1.0 - Hx)
    M = 1.0 + (1.0 + eps) * cost.max(axis=1)

    valid = A >= X[:, None] - 1e-12
    for xi in range(m):
        seen = set()
        for a in range(n+1):
            key = round(float(A[xi, a]), 12)
            if key in seen:
                valid[xi, a] = False
            else:
                seen.add(key)
    return dict(p=priors, t=thresholds, tt=tt, c=c, n=n, m=m, X=X, D=D, A=A, cost=cost, Hx=Hx, U=U, L=L, M=M, eps=eps, valid=valid)

In [6]:
def build_variables(K):
    n, m = K["n"], K["m"]

    xv = cp.Variable((n, n), boolean=True, name="x")
    yv = cp.Variable((m*n, n+1), boolean=True, name="y")

    cons = [
        cp.sum(xv, axis=1) == 1,
        cp.sum(yv, axis=1) == 1,
    ]
    return xv, yv, cons

def row(x_idx, j, n):
    return x_idx * n + j

def assignment_to_partition(assign):
    bins = {}
    for i, j in enumerate(assign):
        bins.setdefault(j, []).append(i)
    return frozenset(frozenset(v) for v in bins.values())

def bell(n):
    r = [1]
    for _ in range(n):
        new = [r[-1]]
        for v in r:
            new.append(new[-1] + v)
        r = new
    return r[0]

In [7]:
def add_best_response(K, xv, yv):
    n, m, U, M, valid = K["n"], K["m"], K["U"], K["M"], K["valid"]
    ones = np.ones((1, n+1))
    cons = []

    for x_idx in range(m):
        util = xv.T @ U[x_idx]
        y_x = yv[row(x_idx, 0, n): row(x_idx, 0, n)+n, :]

        for a in range(n+1):
            if not valid[x_idx, a]:
                cons.append(y_x[:, a] == 0)
                continue
            u_a = cp.reshape(util[:, a], (n,1), order="C")
            y_a = cp.reshape(y_x[:, a], (n,1), order="C")
            cons.append((u_a + M[x_idx] * (1 - y_a)) @ ones >= util)
    return cons

def chosen_landing(K, yv_value):
    n, m = K["n"], K["m"]
    Y = np.asarray(yv_value).reshape(m, n, n+1)
    a = Y.argmax(axis=2)
    return K["A"][np.arange(m)[:, None], a]

def brute_force_landing(K, assign):
    n, m = K["n"], K["m"]
    out = np.zeros((m, n))
    for j in range(n):
        B = (assign == j).astype(float)
        if B.sum() == 0:
            out[:, j] = np.nan
            continue
        score = (K["U"] + B[None, :, None]).sum(axis=1)
        score = np.where(K["valid"], score, -np.inf)
        out[:, j] = K["A"][np.arange(m), score.argmax(axis=1)]
    return out

In [8]:
def add_objective(K, xv, yv):
    n, m , L, D, p = K["n"], K["m"], K["L"], K["D"], K["p"]

    v = cp.Variable((m * n, n), nonneg=True, name="v")
    cons = []
    for x_idx in range(m):
        sl = slice(x_idx * n, (x_idx + 1) * n)
        g = yv[sl, :] @ L[x_idx].T
        cons.append(v[sl, :] >= g + xv.T - 1)

    W = np.repeat(D, n)[:, None] * p[None, :]
    return cp.sum(cp.multiply(W, v)), cons

def evaluate_partition(K, assign):
    n, m, U, L, D, p, valid = K["n"], K["m"], K["U"], K["L"], K["D"], K["p"], K["valid"]

    total = 0.0
    for j in range(n):
        members = np.flatnonzero(assign == j)
        if members.size == 0:
            continue
        B = np.zeros(n)
        B[members] = 1.0
        score = (U * B[None, :, None]).sum(axis=1)
        a_star = np.where(valid, score, -np.inf).argmax(axis=1)
        err = L[np.arange(m)[:, None], members[None, :], a_star[:, None]]
        total += float(D @ (err @ p[members]))
    return total

def add_symmetry(K, xv):
    n = K["n"]
    cons = [xv[i, j] == 0 for i in range(n) for j in range(i+1, n)]
    for i in range(1, n):
        for j in range(1, i+1):
            cons.append(xv[i,j] <= cp.sum(xv[:i, j-1]))
    return cons

def pin_empty_bins(K, xv, yv):
    n = K["n"]
    occ = cp.sum(xv, axis=0)
    return [yv[x_idx * n:(x_idx + 1) * n, 0] >= 1 - occ for x_idx in range(K["m"])]

def solve_milp(priors, thresholds, tt, c, m=51, eps=1e-6, reduce=True, pin_empty=True, feas_tol=1e-9, check_tol=1e-9, time_limit=None, solver=cp.GUROBI):
    K = build_constants(priors, thresholds, tt, c, m, eps)
    xv, yv, cons = build_variables(K)
    cons = cons + add_best_response(K, xv, yv)
    if reduce:
        cons = cons + add_symmetry(K, xv)
    if pin_empty:
        cons = cons + pin_empty_bins(K, xv, yv)
    obj, obj_cons = add_objective(K, xv, yv)

    prob = cp.Problem(cp.Minimize(obj), cons + obj_cons)
    if str(solver).upper() == 'GUROBI':
        kw = {"FeasibilityTol": feas_tol, "IntFeasTol": feas_tol}
        if time_limit is not None:
            kw["TimeLimit"] = time_limit
    else:
        kw = dict(primal_feasibility_tolerance=feas_tol, mip_feasibility_tolerance=feas_tol)
        if time_limit is not None:
            kw["time_limit"] = time_limit
    
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob.solve(solver=solver, **kw)
    elapsed = time.perf_counter() - t0

    assign = np.asarray(xv.value).argmax(axis=1)
    loss = evaluate_partition(K, assign)
    return dict(
        status=prob.status,
        partition=sorted(sorted(b) for b in assignment_to_partition(assign)),
        loss=loss,
        solver_objective=float(prob.value),
        certified=bool(loss - float(prob.value) < check_tol),
        seconds=elapsed,
        K=K
    )

In [ ]:
def solve_miqp(priors, thresholds, true_threshold, c, m=101, eps=1e-6, reduce=False, pin_empty=True, time_limit=None, quiet=True, check_tol=1e-9, threads=None): 
    K = build_constants(priors, thresholds, true_threshold, c, m, eps)
    n, m = K["n"], K["m"]
 
    U, L, D, p, M, valid = K["U"], K["L"], K["D"], K["p"], K["M"], K["valid"]
 
    env = gp.Env(empty=True)
    if quiet:
        env.setParam("OutputFlag", 0)
    env.start()
    mdl = gp.Model("partition_miqp", env=env)
 
    mdl.setParam("FeasibilityTol", 1e-9)
    mdl.setParam("IntFeasTol", 1e-9)
    if threads is not None:
        mdl.setParam("Threads", int(threads))
    if time_limit is not None:
        mdl.setParam("TimeLimit", float(time_limit))
 
    xb = mdl.addVars(n, n, vtype=GRB.BINARY, name="x")
    yb = mdl.addVars(m, n, n + 1, vtype=GRB.BINARY, name="y")
 
    for i in range(n):
        mdl.addConstr(gp.quicksum(xb[i, j] for j in range(n)) == 1)
 
    for xi in range(m):
        for j in range(n):
            mdl.addConstr(gp.quicksum(yb[xi, j, a] for a in range(n + 1)) == 1)
 
    for xi in range(m):
        for a in range(n + 1):
            if not valid[xi, a]:
                for j in range(n):
                    mdl.addConstr(yb[xi, j, a] == 0)
 
    for xi in range(m):
        acts = [a for a in range(n + 1) if valid[xi, a]]
        for j in range(n):
            for a in acts:
                ua = gp.quicksum(float(U[xi, i, a]) * xb[i, j] for i in range(n))
                for a2 in acts:
                    if a2 == a:
                        continue
                    ua2 = gp.quicksum(float(U[xi, i, a2]) * xb[i, j]
                                      for i in range(n))
                    mdl.addConstr(ua >= ua2 - float(M[xi]) * (1 - yb[xi, j, a]))
 
    if reduce:
        for i in range(n):
            for j in range(i + 1, n):
                mdl.addConstr(xb[i, j] == 0)
            for i in range(1, n):
                for j in range(1, i + 1):
                    mdl.addConstr(xb[i, j] <=
                                  gp.quicksum(xb[k, j - 1] for k in range(i)))
 
    if pin_empty:
        for xi in range(m):
            for j in range(n):
                occ = gp.quicksum(xb[i, j] for i in range(n))
                mdl.addConstr(yb[xi, j, 0] >= 1 - occ)
 
    obj = gp.QuadExpr()
    for xi in range(m):
        for a in range(n + 1):
            if not valid[xi, a]:
                continue
            for i in range(n):
                coef = float(D[xi] * p[i] * L[xi, i, a])
                if coef == 0.0:
                    continue
                for j in range(n):
                    obj.add(xb[i, j] * yb[xi, j, a], coef)
    mdl.setObjective(obj, GRB.MINIMIZE)
 
    t0 = time.perf_counter()
    mdl.optimize()
    elapsed = time.perf_counter() - t0
 
    out = dict(
        seconds=elapsed, 
        status=mdl.Status,
        proven_optimal=(mdl.Status == GRB.OPTIMAL),
        dual_bound=(mdl.ObjBound if mdl.SolCount or mdl.Status != GRB.LOADED else None),
        n_vars=mdl.NumVars, n_cons=mdl.NumConstrs,
        n_qterms=mdl.NumQNZs, 
        K=K
        )
 
    if mdl.SolCount == 0:
        out.update(assign=None, partition=None, loss=None,
                   solver_objective=None, certified=False)
        return out
 
    Xm = np.array([[xb[i, j].X for j in range(n)] for i in range(n)])
    assign = Xm.argmax(axis=1)
    loss = evaluate_partition(K, assign)
    bins = {}
    for i, j in enumerate(assign):
        bins.setdefault(int(j), []).append(i)
    out.update(assign=assign,
               partition=sorted(sorted(v) for v in bins.values()),
               loss=loss, solver_objective=float(mdl.ObjVal),
               certified=bool(loss - float(mdl.ObjVal) < check_tol))
    return out

In [10]:
def add_result(results, alg, time, loss, ratio, partition, priors, thresholds, c, tt, m, n):
    results["alg"].append(alg)
    results["time"].append(time)
    results["loss"].append(loss)
    results["ratio"].append(ratio)
    results["partition"].append(partition)
    results["priors"].append(priors)
    results["thresholds"].append(thresholds)
    results["c"].append(c)
    results["tt"].append(tt)
    results["m"].append(m)
    results["n"].append(n)

def run_example(priors, thresholds, tt, c, m, run_opt=True, run_lp=True, run_qp=True, run_unreduced=True, time_limit=None):
    n = len(thresholds)
    K = build_constants(priors, thresholds, tt, c, m)
    fl = 4*(n+1)
    fh = (fl-11)//2

    results = {"alg": [], "time": [], "loss": [], "ratio": [], "partition": [], "priors": [], "thresholds": [], "c": [], "tt": [], "m": [], "n": []}
    p_agg, loss_agg, time_agg = find_partitions_greedy_agg(K["X"], thresholds, priors, tt, c)
    p_div, loss_div, time_div = find_partitions_greedy_div(K["X"], thresholds, priors, tt, c)
    if run_opt:
        p_opt, loss_opt, time_opt = find_partitions_optimal(K["X"], thresholds, priors, tt, c)
    
        r_opt = 1.0
        r_agg = approximation_ratio(loss_opt, loss_agg)
        r_div = approximation_ratio(loss_opt, loss_div)
    else:
        r_agg = np.nan
        r_div = np.nan

    if run_opt:
        add_result(results, "OPT", time_opt, loss_opt, r_opt, p_opt, priors, thresholds, c, tt, m, n)
    add_result(results, "AGG", time_agg, loss_agg, r_agg, p_agg, priors, thresholds, c, tt, m, n)
    add_result(results, "DIV", time_div, loss_div, r_div, p_div, priors, thresholds, c, tt, m, n)
    

    print(f"m                  : {m}")
    print(f"n                  : {n}")
    print(f"c                  : {c}")
    print(f"t*                 : {tt:.4f}")
    with np.printoptions(formatter={'float': '{: 0.4f}'.format}):
        print(f"priors             : {priors}")
        print(f"thresholds         : {thresholds}\n")

    print(f"{'Algorithm':^12}  |  {'Time':^12}  |  {'Loss':^12}  |  {'Ratio':^12}  |  {' '*fh}{'Partition'}{' '*fh}")
    print(f"{'='*14}|{'='*16}|{'='*16}|{'='*16}|{'='*fl}")
    for i in range(len(results["alg"])):
        print(f"{results["alg"][i]:^12}  |  {results["time"][i]:11.2f}s  |  {results["loss"][i]:12.3f}  |  {results["ratio"][i]:12.3f}  |  {results["partition"][i]}")
    
    if run_lp:
        lp_algs = [("LP_R", True)]
        if run_unreduced:
            lp_algs.append(("LP", False))
        for name, reduce in lp_algs:
            if isinstance(time_limit, float) or isinstance(time_limit, int) or time_limit is None:
                time_limit = [time_limit]
            for tl in time_limit:
                lp = solve_milp(priors, thresholds, tt, c, m, reduce=reduce, time_limit=tl)
                p_lp, loss_lp, time_lp = lp["partition"], lp["loss"], lp["seconds"]
                if run_opt:
                    r_lp = approximation_ratio(loss_opt, loss_lp)
                else:
                    r_lp = np.nan
                alg_name = f"{name}-{tl}s" if tl else name
                print(f"{alg_name:^12}  |  {time_lp:11.2f}s  |  {loss_lp:12.3f}  |  {r_lp:12.3f}  |  {p_lp}")
                add_result(results, alg_name, time_lp, loss_lp, r_lp, p_lp, priors, thresholds, c, tt, m, n)
    
    if run_qp:
        qp_algs = [("QP_R", True)]
        if run_unreduced:
            qp_algs.append(("QP", False))
        for name, reduce in qp_algs:
            if isinstance(time_limit, float) or isinstance(time_limit, int) or time_limit is None:
                time_limit = [time_limit]
            for tl in time_limit:
                qp = solve_miqp(priors, thresholds, tt, c, m, reduce=reduce, time_limit=tl)
                p_qp, loss_qp, time_qp = qp["partition"], qp["loss"], qp["seconds"]
                if run_opt:
                    r_qp = approximation_ratio(loss_opt, loss_qp)
                else:
                    r_qp = np.nan
                alg_name = f"{name}-{tl}s" if tl else name
                print(f"{alg_name:^12}  |  {time_qp:11.2f}s  |  {loss_qp:12.3f}  |  {r_qp:12.3f}  |  {p_qp}")
                add_result(results, alg_name, time_qp, loss_qp, r_qp, p_qp, priors, thresholds, c, tt, m, n)
    
    return results

### Example 1

In [ ]:
m = 21
n = 9
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, run_opt=True, run_unreduced=True)

m                  : 21
n                  : 9
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111  0.1111]
thresholds         : [ 0.0000  0.1250  0.2500  0.3750  0.5000  0.6250  0.7500  0.8750  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |                Partition              
==============|================|================|================|========================================
    AGG       |         0.03s  |         0.053  |           nan  |  [[2], [0, 4, 6], [1, 5], [3, 7, 8]]
    DIV       |         0.04s  |         0.058  |           nan  |  [[0, 2, 3, 4, 5, 6, 7], [1], [8]]
    LP_R      |         1.39s  |         0.042  |           nan  |  [[0, 3, 4, 6, 7], [1, 2], [5, 8]]
     LP       |         3.60s  |         0.042  |           nan  |  [[0, 3, 4, 6, 7], [1, 2], [5, 8]]
    QP_R      |         0.93s  |         0.042  |           nan  |  [[0, 3, 4, 6, 7], 

### Example 2

In [211]:
m = 501
n = 5
p1 = 0.02/0.98
p2 = 0.02/0.98
diff = 1 - p1 - p2
priors = np.array([0.14 * diff/0.94, p1, p2, 0.56 * diff/0.94, 0.24 * diff/0.94])
thresholds = np.linspace(0.0, 1.0, n)
c = 1.
tt = 0.1489
assert(abs(np.sum(priors) - 1) < 1e-6)

res2 = run_example(priors, thresholds, tt, c, m, run_unreduced=False)

m                  : 501
n                  : 5
c                  : 1.0
t*                 : 0.1489
priors             : [ 0.1429  0.0204  0.0204  0.5714  0.2449]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |        Partition      
==============|================|================|================|========================
    OPT       |         0.01s  |         0.027  |         1.000  |  [[1, 2], [0, 3, 4]]
    AGG       |         0.01s  |         0.145  |         5.290  |  [[0, 1], [2, 3, 4]]
    DIV       |         0.01s  |         0.145  |         5.290  |  [[2, 3, 4], [0, 1]]
    LP_R      |        11.31s  |         0.027  |         1.000  |  [[0, 3, 4], [1, 2]]
    QP_R      |         2.03s  |         0.027  |         1.000  |  [[0, 3, 4], [1], [2]]


### Example 3

In [226]:
m = 501
n = 5
p = (0.2444/0.25 - 0.97) / (0.4888/0.25 - 0.97)
diff = 1-2*p
priors = np.array([0.097*diff/0.97, p, p, 0.6286*diff/0.97, 0.2444*diff/0.97])
thresholds = np.linspace(0.0, 1.0, n)
c = 1.
tt = 0.1

res3 = run_example(priors, thresholds, tt, c, m, run_unreduced=True)

m                  : 501
n                  : 5
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.0985  0.0077  0.0077  0.6380  0.2481]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |        Partition      
==============|================|================|================|========================
    OPT       |         0.01s  |         0.013  |         1.000  |  [[1, 2], [0, 3, 4]]
    AGG       |         0.02s  |         0.013  |         1.000  |  [[0, 3, 4], [1, 2]]
    DIV       |         0.01s  |         0.098  |         7.450  |  [[2, 3, 4], [0, 1]]
    LP_R      |        17.04s  |         0.013  |         1.000  |  [[0, 3, 4], [1], [2]]
     LP       |        41.30s  |         0.013  |         1.000  |  [[0, 3, 4], [1, 2]]
    QP_R      |         3.08s  |         0.013  |         1.000  |  [[0, 3, 4], [1], [2]]
     QP       |        15.72s  |         0.013  |         1

### Hard Example for Greedy

In [ ]:
m = 501
# m = 200001
n = 5

c = 1.
tt = 0.005

p = 0.0032
p0 = tt*(1-p)
p1 = p2 = p/2
p4 = 0.25*(1 - 0.75*p)
p3 =1 - p0 - p - p4

priors = np.array([p0,p1,p2,p3,p4])
thresholds = np.linspace(0.0, 1.0, n)

res3 = run_example(priors, thresholds, tt, c, m, run_unreduced=False)

m                  : 501
n                  : 5
c                  : 1.0
t*                 : 0.0050
priors             : [ 0.0050  0.0016  0.0016  0.7424  0.2494]
thresholds         : [ 0.0000  0.2500  0.5000  0.7500  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |        Partition      
==============|================|================|================|========================
    OPT       |         0.01s  |         0.000  |         1.000  |  [[1, 2], [0, 3, 4]]
    AGG       |         0.01s  |         0.000  |         1.000  |  [[0, 3, 4], [1, 2]]
    DIV       |         0.01s  |         0.004  |        81.598  |  [[2, 3, 4], [0, 1]]
    LP_R      |        17.72s  |         0.000  |         1.000  |  [[0, 3, 4], [1], [2]]
    QP_R      |         3.05s  |         0.000  |         1.000  |  [[0, 3, 4], [1], [2]]


### Large Number of Classifiers

In [12]:
m = 51
n = 15
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, run_opt=False, run_unreduced=False)

m                  : 51
n                  : 15
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667
  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667]
thresholds         : [ 0.0000  0.0714  0.1429  0.2143  0.2857  0.3571  0.4286  0.5000  0.5714
  0.6429  0.7143  0.7857  0.8571  0.9286  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |                            Partition                          
==============|================|================|================|================================================================
    AGG       |         0.18s  |         0.030  |           nan  |  [[1, 9], [0, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14], [2, 3]]
    DIV       |         0.04s  |         0.025  |           nan  |  [[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], [2], [14]]
    LP_R      |       119.77s  |         0.025  |           nan  |  [[0, 3, 4, 5, 6, 8, 9, 10, 11, 

In [13]:
m = 51
n = 15
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, run_opt=False, run_unreduced=False, time_limit=[25, 50, 75, 100, None])

m                  : 51
n                  : 15
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667
  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667]
thresholds         : [ 0.0000  0.0714  0.1429  0.2143  0.2857  0.3571  0.4286  0.5000  0.5714
  0.6429  0.7143  0.7857  0.8571  0.9286  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |                            Partition                          
==============|================|================|================|================================================================
    AGG       |         0.12s  |         0.030  |           nan  |  [[1, 9], [0, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14], [2, 3]]
    DIV       |         0.09s  |         0.025  |           nan  |  [[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], [2], [14]]
  LP_R-25s    |        40.58s  |         0.056  |           nan  |  [[0, 2, 3, 6, 7], [1, 4, 5, 8, 

In [ ]:
m = 101
n = 15
c = 1.
tt = 0.1
thresholds = np.linspace(0.0, 1.0, n)
priors = np.full(n, 1/n)

res1 = run_example(priors, thresholds, tt, c, m, run_opt=False, run_unreduced=False, time_limit=[100, None])

m                  : 101
n                  : 15
c                  : 1.0
t*                 : 0.1000
priors             : [ 0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667
  0.0667  0.0667  0.0667  0.0667  0.0667  0.0667]
thresholds         : [ 0.0000  0.0714  0.1429  0.2143  0.2857  0.3571  0.4286  0.5000  0.5714
  0.6429  0.7143  0.7857  0.8571  0.9286  1.0000]

 Algorithm    |      Time      |      Loss      |     Ratio      |                            Partition                          
==============|================|================|================|================================================================
    AGG       |         0.15s  |         0.059  |           nan  |  [[14], [0, 9, 10], [1, 4, 8, 11, 12, 13], [2, 3, 5, 6, 7]]
    DIV       |         0.05s  |         0.032  |           nan  |  [[0, 1, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13], [2, 6], [14]]
 LP_R-100s    |       137.65s  |         0.097  |           nan  |  [[0], [1], [2], [3], [4], [5